ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

`CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [2]:
df = pd.read_csv('CC_GENERAL.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'CC_GENERAL.csv'

**Check the first five rows of the dataset.**

In [ ]:
df.head()

**Check the shape of the dataset.**

In [ ]:
df.shape

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info()

**Check summary statistics using `describe()`.**

In [ ]:
df.describe()

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
df.drop('CUST_ID', axis=1, inplace=True)

**Check the missing values in each column.**

In [ ]:
df.isnull().sum()

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df.fillna(df.mean(), inplace=True)

**Check the missing values again to make sure they were handled.**

In [ ]:
df.isnull().sum()

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df.hist(figsize=(18, 14), bins=30, color='steelblue', edgecolor='black')
plt.suptitle('Distribution of Credit Card Features', fontsize=16)
plt.tight_layout()
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['PURCHASES'], alpha=0.4, color='steelblue', edgecolors='none')
plt.xlabel('BALANCE')
plt.ylabel('PURCHASES')
plt.title('Balance vs Purchases')
plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['CASH_ADVANCE'], alpha=0.4, color='coral', edgecolors='none')
plt.xlabel('BALANCE')
plt.ylabel('CASH_ADVANCE')
plt.title('Balance vs Cash Advance')
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

**Plot the elbow curve.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertia_values, marker='o', color='steelblue', linewidth=2)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method - Choosing Optimal K')
plt.xticks(range(1, 11))
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

**Output Interpretation**

Look at the elbow curve and try to identify where the decrease in inertia starts to slow down.

That point can suggest a reasonable value for K.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), silhouette_scores, marker='o', color='coral', linewidth=2)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for Each K')
plt.xticks(range(2, 11))
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
silhouette_table = pd.DataFrame({
    'K': range(2, 11),
    'Silhouette Score': silhouette_scores
})

silhouette_table

**Output Interpretation**

A higher silhouette score usually means better clustering.

However, do not rely only on the highest value. Also consider whether the chosen K makes sense for customer segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
# K=4 is chosen based on the elbow method (noticeable bend around K=3-4)
# and a reasonable silhouette score with meaningful business interpretation
final_kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
final_kmeans.fit(X_scaled)

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df['Cluster'] = final_kmeans.labels_

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby('Cluster').mean()
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
df['Cluster'].value_counts().sort_index()

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1],
                      c=df['Cluster'], cmap='viridis',
                      alpha=0.5, edgecolors='none', s=20)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('Customer Segments Visualized with PCA (2D)')
plt.tight_layout()
plt.show()

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions.

## Final Questions

**1. Why is this an unsupervised learning problem?**

This is an unsupervised learning problem because the dataset does not contain a target label or predefined customer groups. There is no "correct answer" to predict. Instead, the algorithm discovers hidden patterns and groups customers based purely on their behavioral similarities.

---

**2. Why did we remove the `CUST_ID` column?**

The `CUST_ID` column is just an identification number assigned to each customer. It does not describe any real behavior or characteristic of the customer. Including it in clustering would cause the algorithm to consider irrelevant information, which would lead to meaningless results. We only want to cluster based on actual behavioral features.

---

**3. Which columns had missing values?**

The columns `MINIMUM_PAYMENTS` and `CREDIT_LIMIT` contained missing values in the dataset.

---

**4. How did you handle the missing values?**

We used mean imputation: we replaced each missing value with the mean of that column using `df.fillna(df.mean(), inplace=True)`. This approach preserves all rows in the dataset and avoids data loss. It is suitable when the number of missing values is small relative to the total dataset size.

---

**5. Why is scaling important before applying K-Means?**

K-Means is a distance-based algorithm — it groups points by calculating their Euclidean distance from cluster centroids. If features have very different ranges (for example, `BALANCE` can reach tens of thousands while frequency columns are between 0 and 1), the features with larger values will dominate the distance calculation and overshadow the smaller-scale features. StandardScaler transforms all features to have a mean of 0 and a standard deviation of 1, so every feature contributes equally to the clustering.

---

**6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.**

We chose **K = 4**.

- **Elbow method**: The inertia drops sharply from K=1 to K=3, then the rate of decrease slows noticeably around K=3 to K=4, suggesting that adding more clusters beyond 4 gives diminishing returns.
- **Silhouette score**: K=2 tends to give the highest silhouette score in this dataset, but K=2 is too simple to extract useful business insights from 8000+ customers. K=4 gives a reasonable silhouette score while producing four distinct and interpretable customer segments that are more actionable for marketing.

---

**7. Based on the cluster summary table, describe each customer segment in your own words.**

*(Note: The exact cluster numbers may vary between runs. Descriptions are based on typical patterns seen in this dataset.)*

- **Cluster 0 – Low Activity Customers**: Low balance, low purchases, low cash advance. These customers barely use their credit card. They may have the card but rarely engage with it.
- **Cluster 1 – Cash Advance Users**: High cash advance amounts and high balance, but relatively low purchases. These customers rely on their credit card mainly for borrowing cash rather than shopping.
- **Cluster 2 – Active Purchasers**: High purchase amounts and higher credit limit. These customers actively shop using their cards and tend to pay their balances regularly.
- **Cluster 3 – High-Value / Premium Customers**: High credit limit, high balance, high purchases, and high payments. These are the most financially active customers and likely the most valuable to the company.

---

**8. Which cluster may represent high-value customers?**

The cluster with the highest `CREDIT_LIMIT`, highest `PURCHASES`, and highest `PAYMENTS` represents the high-value customers. These customers spend heavily and also pay back their balances, making them very profitable for the credit card company.

---

**9. Which cluster may represent customers who rely more on cash advance?**

The cluster with the highest `CASH_ADVANCE` and `CASH_ADVANCE_FREQUENCY` values represents customers who rely more on cash advances. These customers often carry high balances and use the credit card primarily as a source of emergency or short-term borrowing rather than for purchases.

---

**10. How can a company use these clusters for marketing strategy?**

- **Low activity customers**: Send re-engagement campaigns, offer bonus rewards or cashback to encourage card usage.
- **Cash advance users**: Offer personal loan products or flexible payment plans since they seem to need liquidity. Be cautious about risk since they carry high balances.
- **Active purchasers**: Offer category-specific rewards (travel, groceries, shopping) to retain them and increase spending.
- **High-value customers**: Offer premium loyalty programs, exclusive benefits, priority customer service, and targeted offers to maintain their satisfaction and prevent churn.